# HW2 Playground

Fill in TODOs as you work through the assignment.
Implement the required sections in `model.py`, and use this notebook to orchestrate and run your solution.

In [35]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from hw2_loader import HW2DataLoader
from model import GradientBoostingModel

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
# TODO: Load both datasets
loader = HW2DataLoader()

# Heart disease dataset
heart_path = Path('../data/heart.csv')
X_heart, y_heart = loader.get_heart_disease_data(csv_path=heart_path)
print(X_heart.shape, y_heart.value_counts().to_dict())

# Cancer genomics dataset
cancer_path = Path('../data/cancer_genomics.csv')
labels_path = Path('../data/labels_cancer_genomics.csv')
X_cancer, y_cancer = loader.get_cancer_genomics_data(
    csv_path=cancer_path, labels_path=labels_path
)
print(X_cancer.shape, y_cancer.value_counts().to_dict())


Successfully loaded heart disease data with 1025 rows
(1025, 13) {1: 526, 0: 499}
(801, 5479) {'BRCA': 300, 'KIRC': 146, 'LUAD': 141, 'PRAD': 136, 'COAD': 78}


In [37]:
# TODO: Initialize your model (adjust params)
model = GradientBoostingModel(
    task='regression',
    max_depth=3,
    learning_rate=0.1,
    n_estimators=100,
    use_scaler=True,
)


In [38]:
# TODO: Train/test split + fit (heart)
X_train, X_test, y_train, y_test = model.train_test_split(X_heart, y_heart)
#X_train, X_test, y_train, y_test = model.train_test_split(heart_data.drop("target", axis=1), heart_data["target"])
model.fit(X_train, y_train)


In [39]:
# TODO: Evaluate (heart)
metrics = model.evaluate(X_test, y_test)
print(metrics)



{'rmse': 0.07407866489977881, 'mae': 0.19033021312493176, 'r2': 0.7036782893191315}


In [40]:
# TODO: Cross-validation (heart)
cv_results = model.cross_validate(X_heart, y_heart)
print(cv_results)

{'neg_mean_squared_error': {'mean': np.float64(-0.052299046840056075), 'std': np.float64(0.007316239159884808)}, 'neg_mean_absolute_error': {'mean': np.float64(-0.1618476777967438), 'std': np.float64(0.008614284113473378)}, 'r2': {'mean': np.float64(0.7902840379185034), 'std': np.float64(0.02909267252824465)}}


In [ ]:
# TODO: Feature importance (heart)
model.get_feature_importance(plot=False)
feature_importance = model.get_feature_importance()
model.plot_tree()
print(feature_importance.head(10))

    feature  importance
0       age    0.057732
1       sex    0.022709
2        cp    0.266662
3  trestbps    0.045744
4      chol    0.060002
5       fbs    0.000174
6   restecg    0.001890
7   thalach    0.053058
8     exang    0.020264
9   oldpeak    0.108313


In [42]:
# TODO: Hyperparameter tuning (heart)
param_grid = {
    'max_depth': [3, 5, 7],
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
}
tuning_results = model.tune_hyperparameters(X_heart, y_heart, param_grid, cv=3)
print(tuning_results['best_params'])
print(tuning_results['best_score'])


/opt/miniconda3/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:927: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/opt/miniconda3/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 916, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "/opt/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_scorer.py", line 317, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_scorer.py", line 408, in _score
    response_method = _check_response_method(estimator, self._response_method)
  File "/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py", line 2261, in _check_response_method
    r

{'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100}
nan


/opt/miniconda3/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:927: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/opt/miniconda3/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 916, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "/opt/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_scorer.py", line 317, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_scorer.py", line 408, in _score
    response_method = _check_response_method(estimator, self._response_method)
  File "/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py", line 2261, in _check_response_method
    r

In [43]:
# TODO: Train/evaluate on cancer dataset (multi-class)
cancer_model = GradientBoostingModel()
X_train, X_test, y_train, y_test = cancer_model.train_test_split(
    X_cancer, y_cancer, test_size = 0.2, random_state = 42)
cancer_model.fit(X_train, y_train)
metrics = cancer_model.evaluate(X_test, y_test)
print(metrics)

{'accuracy': 0.9937888198757764, 'precision': 0.9938890002003605, 'recall': 0.9937888198757764, 'f1': 0.993759584539291}
